# Project 4 — Trợ lý hỏi–đáp RAG trên văn bản pháp quy FMCG
## Phần 1: Thu thập dữ liệu & Chunking

**Bối cảnh.** Xây dựng trợ lý trả lời câu hỏi về quy định pháp luật áp dụng cho vận hành chuỗi cung ứng FMCG: nhập nguyên liệu (hải quan) → sản xuất (an toàn thực phẩm) → ra thị trường (ghi nhãn). Yêu cầu cốt lõi: câu trả lời phải **kèm trích dẫn nguồn** đến từng Điều/Khoản của văn bản gốc.

**Dữ liệu.** 5 văn bản quy phạm pháp luật công khai, ưu tiên bản hợp nhất để tránh trích dẫn điều khoản đã hết hiệu lực: Luật An toàn thực phẩm (VBHN 61/2025), Luật Hải quan (VBHN 54/2026), NĐ 15/2018, NĐ 43/2017 và NĐ 111/2021 (sửa đổi NĐ 43).

**Nội dung notebook này.**
1. Khảo sát định dạng nguồn (PDF text-based vs scan, `.doc` vs `.docx`) và trích xuất văn bản.
2. Tách I/O khỏi logic: `read_docx_paragraphs` / `read_html_paragraphs` cùng đổ về một hàm chunking duy nhất.
3. Chunking **theo cấu trúc văn bản pháp luật** (Chương → Điều), giữ metadata đầy đủ để phục vụ trích dẫn nguồn.
4. Xử lý các trường hợp đặc biệt: nghị định sửa đổi, điều khoản đã bãi bỏ, điều khoản quá dài.
5. Kiểm soát giới hạn token của mô hình embedding (cắt đệ quy Điều → Khoản → Điểm + gom nhóm).

**Đầu ra:** `data/chunks.json` — 291 chunk, không chunk nào vượt giới hạn 512 token.

> Chi tiết lý thuyết và các bẫy đã gặp: xem `Tong_hop_kien_thuc_4.md`.

In [1]:
# %pip install pymupdf python-docx
# %pip install sentence-transformers

### 1. Xử lý văn bản pháp luật

In [2]:
from docx import Document
from bs4 import BeautifulSoup
import pandas as pd
import re
import random
import copy


In [3]:
def read_docx_paragraphs(path):
    """Đọc file docx, trả về list paragraph của phần toàn văn"""
    doc = Document(path)
    paragraphs = []

    for para in doc.paragraphs:
        text = para.text.strip()
        if text:
            paragraphs.append(text)
    
    return paragraphs

def read_html_paragraphs(path):
    """Đọc file HTML (lưu từ thuvienphapluat.vn), trả về list paragraph
    của phần toàn văn — tương đương list paragraphs đọc từ docx."""
    html = open(path, encoding='utf-8', errors='ignore').read()
    soup = BeautifulSoup(html, 'html.parser')

    # Khoanh vùng khối chứa toàn văn tiếng Việt, không lấy cả trang
    content = soup.select_one('div.cldivContentDocVn') or soup.select_one('#divContentDoc')

    paras = [p.get_text(' ', strip=True) for p in content.find_all('p')]
    paras = [re.sub(r'\s+', ' ', t) for t in paras if t]
    return paras

def chunk_paragraphs(paragraphs, meta):
    """
    Hàm phân chia văn bản luật thành các chunk (mỗi Điều là một chunk)
    
    Args:
        paragraphs (list): Chứa các paragraphs đọc từ file docx/html.
        meta (dict): Chứa các thông tin cố định như bo_luat, so_hieu, ngay_ky, sua_doi, nguon.
        
    Returns:
        list: Danh sách các chunk (dict) chứa thông tin của từng Điều.
    """

    # Khởi tạo list chứa các chunks
    chunks = []
    current = None
    
    # Biến trạng thái lưu thông tin Chương hiện tại
    chuong_so = ""
    chuong_ten = ""
    dang_cho_ten_chuong = False
    
    # Định nghĩa Regex
    # 1. Bắt Chương: Ví dụ "Chương IV"
    pattern_chuong = r"Chương\s+([IVXLCDM]+)"
    
    # 2. Bắt Điều: Ví dụ "Điều 1. Phạm vi điều chỉnh"
    # Nhóm 1 (\d+) bắt số Điều. Nhóm 2 (.*) bắt toàn bộ Tên Điều phía sau dấu chấm.
    pattern_dieu = r"Điều\s+(\d+)\.\s*(.*)"

    # 3. Bắt điểm kết thúc
    pattern_ket_thuc = r"^Nơi nhận:"
    
    for i, para in enumerate(paragraphs):
        para = para.strip()
        
        # Bỏ qua các dòng trống
        if not para:
            continue
        
        # --- LOGIC 1: KIỂM TRA ĐIỀU KIỆN DỪNG ---
        # Kiểm tra xem đoạn này có bắt đầu bằng "Nơi nhận" không
        if re.match(pattern_ket_thuc, para, re.IGNORECASE):
            # Nếu đang có chunk mở thì đóng lại ngay
            if current is not None:
                chunks.append(current)
                current = None
            
            # (Tùy chọn) In ra thông báo để dễ debug
            print(f"Đã cắt bỏ phần rác cuối văn bản từ dòng: '{para}'")
            
            # Thoát hoàn toàn khỏi vòng lặp, không duyệt các paragraph sau nữa
            break

        # --- LOGIC 2: Xử lý dòng chứa tên Chương ---
        # (Nằm ở paragraph kế tiếp ngay sau dòng "Chương X")
        if dang_cho_ten_chuong:
            chuong_ten = para
            dang_cho_ten_chuong = False
            continue
            
        # --- LOGIC 3 Khớp regex Chương ---
        match_chuong = re.match(pattern_chuong, para)
        if match_chuong:
            chuong_so = match_chuong.group(1)
            dang_cho_ten_chuong = True # Bật cờ để lấy dòng tiếp theo làm tên chương
            continue
            
        # --- LOGIC 4: Khớp regex Điều ---
        match_dieu = re.match(pattern_dieu, para)
        if match_dieu:
            # (a) Nếu current khác None thì append nó vào chunks (đóng chunk cũ)
            if current is not None:
                chunks.append(current)
                
            # Lấy thông tin từ các nhóm bắt giữ
            so_dieu = match_dieu.group(1)
            ten_dieu = match_dieu.group(2)
            
            # (b) Mở dict mới để lưu chunk hiện tại
            current = {
                "chunk_id": f"{meta['so_hieu']}_Điều_{so_dieu}",
                "noi_dung": "", 
                "so_dieu": so_dieu,
                "ten_dieu": ten_dieu,
                "so_chuong": chuong_so,
                "ten_chuong": chuong_ten,
                "bo_luat": meta.get("bo_luat", ""),
                "so_hieu": meta.get("so_hieu", ""),
                "ngay_ky": meta.get("ngay_ky", ""),
                "sua_doi": meta.get("sua_doi", ""),
                "nguon": meta.get("nguon", ""),
                "part": ""
            }
            continue
            
        # --- LOGIC 5: Không khớp Chương, cũng không khớp Điều ---
        # Đây là phần nội dung chi tiết của Điều
        if current is not None:
            # Nếu current đang mở: nối para vào text của nó (thêm "\n")
            if current["noi_dung"] == "":
                current["noi_dung"] = para
            else:
                current["noi_dung"] += "\n" + para
        else:
            # Nếu current là None (đang ở phần Mở đầu/Căn cứ pháp lý trước Điều 1): bỏ qua
            pass

    # --- Hết vòng lặp: Xử lý Điều cuối cùng ---
    # Đóng chunk cuối cùng vì không còn "Điều tiếp theo" để kích hoạt lệnh append
    if current is not None:
        chunks.append(current)
        
    return chunks

#### 1.1 Luật An toàn thực phẩm (61/VBHN-VPQH/2025)

In [4]:
file_path_attp = 'data/luat_attp_vbhn61_2025.docx'
para_attp = read_docx_paragraphs(file_path_attp)
doc = Document(file_path_attp)

In [5]:
para_attp[:20]

['VĂN BẢN PHÁP LUẬT KHÁC',
 'VĂN BẢN HỢP NHẤT - VĂN PHÒNG QUỐC HỘI',
 'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM',
 'Độc lập - Tự do - Hạnh phúc',
 'LUẬT',
 'AN TOÀN THỰC PHẨM',
 'Luật An toàn thực phẩm số 55/2010/QH12 ngày 17 tháng 6 năm 2010 của Quốc hội, có hiệu lực kể từ ngày 01 tháng 7 năm 2011, được sửa đổi, bổ sung bởi:',
 '1. Luật số 28/2018/QH14 ngày 15 tháng 6 năm 2018 của Quốc hội sửa đổi, bổ sung một số điều của 11 luật có liên quan đến quy hoạch, có hiệu lực kể từ ngày 01 tháng 01 năm 2019;',
 '2. Luật Thanh tra số 84/2025/QH15 ngày 25 tháng 6 năm 2025 của Quốc hội, có hiệu lực kể từ ngày 01 tháng 7 năm 2025.',
 'Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 1992 đã được sửa đổi, bổ sung một số điều theo Nghị quyết số 51/2001/QH10;',
 'Quốc hội ban hành Luật An toàn thực phẩm.',
 'Chương I',
 'NHỮNG QUY ĐỊNH CHUNG',
 'Điều 1. Phạm vi điều chỉnh',
 'Luật này quy định về quyền và nghĩa vụ của tổ chức, cá nhân trong bảo đảm an toàn thực phẩm; điều kiện bảo đảm an toàn

In [6]:
len(doc.tables)

1

In [7]:
for table_idx, table in enumerate(doc.tables):
    print(f"--- Đang đọc Bảng {table_idx + 1} ---")
        
    # Lấy dữ liệu của từng hàng
    for row_idx, row in enumerate(table.rows):
        # Lấy text trong từng ô của hàng, xóa khoảng trắng thừa và thay dấu xuống dòng bằng dấu cách
        row_data = [cell.text.strip().replace('\n', ' ') for cell in row.cells]
        print(f"Hàng {row_idx}: {row_data}")
    print("\n")

--- Đang đọc Bảng 1 ---
Hàng 0: ['VĂN PHÒNG QUỐC HỘI  Số: 61/VBHN-VPQH', 'XÁC THỰC VĂN BẢN HỢP NHẤT  Hà Nội, ngày 15 tháng 8 năm 2025 CHỦ NHIỆM  Lê Quang Tùng']




In [8]:
meta_attp = {"bo_luat":"An toàn thực phẩm",
            "so_hieu":"61/VBHN-VPQH",
            "ngay_ky":"15/8/2025",
            "sua_doi":"",
            "nguon":"https://congbao.chinhphu.vn/van-ban/van-ban-hop-nhat-so-61-vbhn-vpqh-45855.htm"}

chunks_attp = chunk_paragraphs(para_attp, meta_attp)

In [9]:
len(chunks_attp)

72

In [10]:
length_attp = pd.Series([len(c['noi_dung']) for c in chunks_attp])
length_attp.describe()

count      72.000000
mean      922.319444
std       755.115378
min         0.000000
25%       439.500000
50%       685.000000
75%      1225.000000
max      4886.000000
dtype: float64

In [11]:
length0 = [c['chunk_id'] for c in chunks_attp if len(c['noi_dung']) == 0]
length0

['61/VBHN-VPQH_Điều_66']

In [12]:
print([c for c in chunks_attp if c['chunk_id'] == '61/VBHN-VPQH_Điều_66'])

[{'chunk_id': '61/VBHN-VPQH_Điều_66', 'noi_dung': '', 'so_dieu': '66', 'ten_dieu': '(được bãi bỏ)', 'so_chuong': 'X', 'ten_chuong': 'QUẢN LÝ NHÀ NƯỚC VỀ AN TOÀN THỰC PHẨM', 'bo_luat': 'An toàn thực phẩm', 'so_hieu': '61/VBHN-VPQH', 'ngay_ky': '15/8/2025', 'sua_doi': '', 'nguon': 'https://congbao.chinhphu.vn/van-ban/van-ban-hop-nhat-so-61-vbhn-vpqh-45855.htm', 'part': ''}]


Điều 66 trong văn bản luật này không có nội dung, chỉ có tiêu đề là "được bãi bỏ". Để cụ thể hơn, ta sẽ gắn nội dung footnote của điều này để thể hiện rõ Điều này được bãi bỏ theo quy định nào.

In [13]:
for chunk in chunks_attp:
    if chunk['chunk_id'] == '61/VBHN-VPQH_Điều_66':
        chunk['noi_dung'] = 'Điều này được bãi bỏ theo quy định tại điểm e khoản 1 Điều 62 của Luật Thanh tra số 84/2025/QH15, có hiệu lực kể từ ngày 01 tháng 7 năm 2025.'
        break

In [14]:
random.sample(chunks_attp, 2)

[{'chunk_id': '61/VBHN-VPQH_Điều_12',
  'noi_dung': '1. Tuân thủ các điều kiện quy định tại Điều 10 của Luật này.\n2. Nguyên liệu ban đầu tạo nên thực phẩm phải bảo đảm an toàn và giữ nguyên các thuộc tính vốn có của nó; các nguyên liệu tạo thành thực phẩm không được tương tác với nhau để tạo ra các sản phẩm gây hại đến sức khỏe, tính mạng con người.\n3. Thực phẩm đã qua chế biến bao gói sẵn phải đăng ký bản công bố hợp quy với cơ quan nhà nước có thẩm quyền trước khi lưu thông trên thị trường.\nChính phủ quy định cụ thể việc đăng ký bản công bố hợp quy và thời hạn của bản đăng ký công bố hợp quy đối với thực phẩm đã qua chế biến bao gói sẵn.',
  'so_dieu': '12',
  'ten_dieu': 'Điều kiện bảo đảm an toàn đối với thực phẩm đã qua chế biến',
  'so_chuong': 'III',
  'ten_chuong': 'ĐIỀU KIỆN BẢO ĐẢM AN TOÀN ĐỐI VỚI THỰC PHẨM',
  'bo_luat': 'An toàn thực phẩm',
  'so_hieu': '61/VBHN-VPQH',
  'ngay_ky': '15/8/2025',
  'sua_doi': '',
  'nguon': 'https://congbao.chinhphu.vn/van-ban/van-ban-hop-

#### 1.2 Luật Hải quan (54/VBHN-VPQH/2026)

In [15]:
file_path_hq = 'data/raw_html/luat_haiquan_vbhn54_2026.html'
para_hq = read_html_paragraphs(file_path_hq)

In [16]:
para_hq[:20]

['VĂN PHÒNG QUỐC HỘI -------',
 'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc ---------------',
 'Số: 54/VBHN-VPQH',
 'Hà Nội, ngày 23 tháng 3 năm 2026',
 'LUẬT',
 'HẢI QUAN',
 'Luật Hải quan số 54/2014/QH13 ngày 23 tháng 6 năm 2014 của Quốc hội, có hiệu lực kể từ ngày 01 tháng 01 năm 2015, được sửa đổi, bổ sung bởi:',
 '1. Luật số 71/2014/QH13 ngày 26 tháng 11 năm 2014 của Quốc hội sửa đổi, bổ sung một số điều của các luật về thuế, có hiệu lực kể từ ngày 01 tháng 01 năm 2015;',
 '2. Luật số 35/2018/QH14 ngày 20 tháng 11 năm 2018 của Quốc hội sửa đổi, bổ sung một số điều của 37 luật có liên quan đến quy hoạch, có hiệu lực kể từ ngày 01 tháng 01 năm 2019;',
 '3. Luật số 07/2022/QH15 ngày 16 tháng 6 năm 2022 của Quốc hội sửa đổi, bổ sung một số điều của Luật Sở hữu trí tuệ, có hiệu lực kể từ ngày 01 tháng 01 năm 2023;',
 '4. Luật số 90/2025/QH15 ngày 25 tháng 6 năm 2025 của Quốc hội sửa đổi, bổ sung một số điều của Luật Đấu thầu, Luật Đầu tư theo phương thức đối tác côn

In [17]:
meta_hq = {"bo_luat":"Hải quan",
            "so_hieu":"54/VBHN-VPQH",
            "ngay_ky":"23/3/2026",
            "sua_doi":"",
            "nguon":"https://thuvienphapluat.vn/van-ban/Thuong-mai/Van-ban-hop-nhat-54-VBHN-VPQH-2026-Luat-Hai-quan-698682.aspx"}

chunks_hq = chunk_paragraphs(para_hq, meta_hq)

Đã cắt bỏ phần rác cuối văn bản từ dòng: 'Nơi nhận: - Văn phòng Chính phủ (để đăng Công báo); - Cục KTVB và Quản lý xử lý VPHC, Bộ TP (để đăng trên CSDL Quốc gia về VBPL); - Cục Quản trị, VPQH (để đăng trên Cổng Thông tin điện tử của Quốc hội); - Vụ Chuyển đổi số, VPQH (để đăng trên trang thông tin nội bộ Intranet); - Lưu: HC, TH.'


In [18]:
length_hq = pd.Series([len(c['noi_dung']) for c in chunks_hq])
length_hq.describe()

count     104.000000
mean     1028.250000
std       799.357931
min        91.000000
25%       442.750000
50%       897.000000
75%      1333.250000
max      5460.000000
dtype: float64

In [19]:
random.sample(chunks_hq, 2)

[{'chunk_id': '54/VBHN-VPQH_Điều_91',
  'noi_dung': '1. Trong việc phòng, chống buôn lậu, vận chuyển trái phép hàng hóa qua biên giới, tổ chức và cá nhân liên quan có quyền:\na) Cung cấp các thông tin, hồ sơ tài liệu và chứng cứ liên quan đến vụ việc vi phạm cho cơ quan hải quan; đề nghị cơ quan hải quan trưng cầu giám định để bảo vệ quyền, lợi ích hợp pháp của mình;\nb) Được bảo vệ bí mật, bảo vệ tính mạng và được hưởng các đãi ngộ theo quy định của pháp luật khi cung cấp thông tin, tố giác, tố cáo về các hành vi buôn lậu, vận chuyển trái phép hàng hóa qua biên giới.\n2. Trong việc phòng, chống buôn lậu, vận chuyển trái phép hàng hóa qua biên giới, tổ chức và cá nhân liên quan có nghĩa vụ:\na) Người điều khiển, người có mặt trên phương tiện vận tải phải chấp hành lệnh dừng phương tiện, khám xét và xuất trình giấy tờ, chứng từ, tài liệu theo yêu cầu của công chức hải quan. Người điều khiển phương tiện vận tải có trách nhiệm cung cấp sơ đồ hầm hàng, chỉ dẫn, mở nơi nghi vấn cất giữ hàng

#### 1.3 Nghị định 43/2017

In [20]:
file_path_nd43 = 'data/raw_html/nd43_2017.html'
para_nd43 = read_html_paragraphs(file_path_nd43)

In [21]:
para_nd43[:20]

['CHÍNH PHỦ -------',
 'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc ---------------',
 'Số: 43/2017/NĐ-CP',
 'Hà Nội, ngày 14 tháng 04 năm 2017',
 'NGHỊ ĐỊNH',
 'VỀ NHÃN HÀNG HÓA',
 'Căn cứ Luật tổ chức Chính phủ ngày 19 tháng 6 năm 2015;',
 'Căn cứ Luật chất lượng sản phẩm, hàng hóa ngày 21 tháng 11 năm 2007;',
 'Căn cứ Luật thương mại ngày 14 tháng 6 năm 2005;',
 'Căn cứ Luật bảo vệ quyền lợi người tiêu dùng ngày 30 tháng 11 năm 2010;',
 'Theo đề nghị của Bộ trưởng Bộ Khoa học và Công nghệ;',
 'Chính phủ ban hành Nghị định về nhãn hàng hóa.',
 'Chương I',
 'NHỮNG QUY ĐỊNH CHUNG',
 'Điều 1. Phạm vi điều chỉnh',
 '1. Nghị định này quy định nội dung, cách ghi và quản lý nhà nước về nhãn đối với hàng hóa lưu thông tại Việt Nam, hàng hóa nhập khẩu.',
 '2. Những hàng hóa sau đây không thuộc phạm vi điều chỉnh của Nghị định này:',
 'a) Bất động sản;',
 'b) Hàng hóa tạm nhập tái xuất; hàng hóa tạm nhập để tham gia hội chợ, triển lãm sau đó tái xuất; hàng hóa quá cảnh, hàng

In [22]:
meta_nd43 = {"bo_luat":"Nghị định về nhãn hàng hóa",
            "so_hieu":"43/2017/NĐ-CP",
            "ngay_ky":"14/4/2017",
            "sua_doi":"111/2021/NĐ-CP",
            "nguon":"https://thuvienphapluat.vn/van-ban/Thuong-mai/Nghi-dinh-43-2017-ND-CP-nhan-hang-hoa-346310.aspx"}

chunks_nd43 = chunk_paragraphs(para_nd43, meta_nd43)

Đã cắt bỏ phần rác cuối văn bản từ dòng: 'Nơi nhận: - Ban Bí thư Trung ương Đảng; - Thủ tướng, các Phó Thủ tướng Chính phủ; - Các bộ, cơ quan ngang bộ, cơ quan thuộc Chính phủ; - HĐND, UBND các tỉnh, thành phố trực thuộc trung ương; - Văn phòng Trung ương và các Ban của Đảng; - Văn phòng Tổng Bí thư; - Văn phòng Chủ tịch nước; - Hội đồng dân tộc và các Ủy ban của Quốc hội; - Văn phòng Quốc hội; - Tòa án nhân dân tối cao; - Viện kiểm sát nhân dân tối cao; - Kiểm toán nhà nước; - Ủy ban Giám sát tài chính Quốc gia; - Ngân hàng Chính sách xã hội; - Ngân hàng Phát triển Việt Nam; - Ủy ban trung ương Mặt trận Tổ quốc Việt Nam; - Cơ quan trung ương của các đoàn thể; - VPCP: BTCN, các PCN, Trợ lý TTg, TGĐ Cổng TTĐT, các Vụ, Cục, đơn vị trực thuộc, Công báo; - Lưu: VT, KGVX (3b). KN'


In [23]:
chunks_nd43[24]

{'chunk_id': '43/2017/NĐ-CP_Điều_25',
 'noi_dung': '1. Bộ trưởng Bộ Khoa học và Công nghệ có trách nhiệm hướng dẫn thực hiện Nghị định này.\n2. Các Bộ trưởng, Thủ trưởng cơ quan ngang bộ, Thủ trưởng cơ quan thuộc Chính phủ, Chủ tịch Ủy ban nhân dân các tỉnh, thành phố trực thuộc trung ương chịu trách nhiệm thi hành Nghị định này./.',
 'so_dieu': '25',
 'ten_dieu': 'Trách nhiệm thi hành',
 'so_chuong': 'IV',
 'ten_chuong': 'ĐIỀU KHOẢN THI HÀNH',
 'bo_luat': 'Nghị định về nhãn hàng hóa',
 'so_hieu': '43/2017/NĐ-CP',
 'ngay_ky': '14/4/2017',
 'sua_doi': '111/2021/NĐ-CP',
 'nguon': 'https://thuvienphapluat.vn/van-ban/Thuong-mai/Nghi-dinh-43-2017-ND-CP-nhan-hang-hoa-346310.aspx',
 'part': ''}

In [24]:
length_nd43 = pd.Series([len(c['noi_dung']) for c in chunks_nd43])
length_nd43.describe()

count      25.000000
mean      953.760000
std       817.932927
min       168.000000
25%       383.000000
50%       588.000000
75%      1340.000000
max      3520.000000
dtype: float64

Lưu ý: Còn các bảng phụ lục chưa được xử lý, hiện tại đang tạm thời chưa có chunk cho bảng phụ lục.

#### 1.4 Nghị định 111/2021

In [25]:
file_path_nd111 = 'data/raw_html/nd111_2021.html'
para_nd111 = read_html_paragraphs(file_path_nd111)

In [26]:
para_nd111[:20]

['CHÍNH PHỦ -------',
 'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc ---------------',
 'Số: 111/2021/NĐ-CP',
 'Hà Nội, ngày 09 tháng 12 năm 2021',
 'NGHỊ ĐỊNH',
 'SỬA ĐỔI, BỔ SUNG MỘT SỐ ĐIỀU NGHỊ ĐỊNH SỐ 43/2017/NĐ-CP NGÀY 14 THÁNG 4 NĂM 2017 CỦA CHÍNH PHỦ VỀ NHÃN HÀNG HÓA',
 'Căn cứ Luật Tổ chức Chính phủ ngày 19 tháng 6 năm 2015, Luật sửa đổi, bổ sung một số điều của Luật Tổ chức Chính phủ và Luật Tổ chức chính quyền địa phương ngày 22 tháng 11 năm 2019;',
 'Căn cứ Luật Chất lượng sản phẩm, hàng hóa ngày 21 tháng 11 năm 2007;',
 'Căn cứ Luật Thương mại ngày 14 tháng 6 năm 2005;',
 'Căn cứ Luật Bảo vệ quyền lợi người tiêu dùng ngày 17 tháng 11 năm 2010;',
 'Theo đề nghị của Bộ trưởng Bộ Khoa học và Công nghệ;',
 'Chính phủ ban hành Nghị định sửa đổi, bổ sung một số điều Nghị định số 43/2017/NĐ-CP ngày 14 tháng 4 năm 2017 của Chính phủ về nhãn hàng hóa.',
 'Điều 1. Sửa đổi, bổ sung một số điều Nghị định số 43/2017/NĐ-CP ngày 14 tháng 4 năm 2017 của Chính phủ về nhãn

In [27]:
meta_nd111 = {"bo_luat":"Nghị định về nhãn hàng hóa",
            "so_hieu":"111/2021/NĐ-CP",
            "ngay_ky":"09/12/2021",
            "sua_doi":"",
            "nguon":"https://thuvienphapluat.vn/van-ban/Thuong-mai/Nghi-dinh-111-2021-ND-CP-sua-doi-Nghi-dinh-43-2017-ND-CP-497099.aspx"}

chunks_nd111 = chunk_paragraphs(para_nd111, meta_nd111)

Đã cắt bỏ phần rác cuối văn bản từ dòng: 'Nơi nhận: - Ban Bí thư Trung ương Đảng; - Thủ tướng, các Phó Thủ tướng Chính phủ; - Các bộ, cơ quan ngang bộ, cơ quan thuộc Chính phủ; - HĐND, UBND các tỉnh, thành phố trực thuộc trung ương; - Văn phòng Trung ương và các Ban của Đảng; - Văn phòng Tổng Bí thư; - Văn phòng Chủ tịch nước; - Hội đồng Dân tộc và các Ủy ban của Quốc hội; - Văn phòng Quốc hội; - Tòa án nhân dân tối cao; - Viện kiểm sát nhân dân tối cao; - Ủy ban Giám sát tài chính Quốc gia; - Kiểm toán nhà nước; - Ngân hàng Chính sách xã hội; - Ngân hàng Phát triển Việt Nam; - Ủy ban trung ương Mặt trận Tổ quốc Việt Nam; - Cơ quan trung ương của các đoàn thể; - VPCP: BTCN, các PCN, Trợ lý TTg, TGĐ Cổng TTĐT, các Vụ, Cục, đơn vị trực thuộc, Công báo; - Lưu: VT, KGVX (2b).'


In [28]:
length_nd111 = pd.Series([len(c['noi_dung']) for c in chunks_nd111])
length_nd111.describe()

count       4.000000
mean     2412.750000
std      4300.024138
min        55.000000
25%       222.250000
50%       369.000000
75%      2559.500000
max      8858.000000
dtype: float64

In [29]:
chunks_nd111[0]

{'chunk_id': '111/2021/NĐ-CP_Điều_1',
 'noi_dung': '1. Sửa đổi, bổ sung Điều 1\n“Điều 1. Phạm vi điều chỉnh\n1. Nghị định này quy định nội dung, cách ghi và quản lý nhà nước về nhãn đối với hàng hóa lưu thông tại Việt Nam, hàng hóa xuất khẩu, nhập khẩu.\n2. Những loại hàng hóa sau đây không thuộc phạm vi điều chỉnh của Nghị định này:\na) Bất động sản;\nb) Hàng hóa tạm nhập tái xuất; hàng hóa quá cảnh, hàng hóa chuyển khẩu; hàng hóa trung chuyển; hàng hóa nhập khẩu gửi kho ngoại quan để xuất khẩu sang nước thứ ba;\nc) Hành lý của người xuất cảnh, nhập cảnh; tài sản di chuyển;\nd) Hàng hóa bị tịch thu bán đấu giá;\nđ) Hàng hóa là thực phẩm tươi, sống, thực phẩm chế biến không có bao bì và bán trực tiếp cho người tiêu dùng;\ne) Hàng hóa là nhiên liệu, nguyên liệu (nông sản, thủy sản, khoáng sản), phế liệu (trong sản xuất, kinh doanh), vật liệu xây dựng không có bao bì và được bán trực tiếp cho người tiêu dùng;\ng) Hàng hóa là xăng dầu, khí (LPG, CNG, LNG) chất lỏng, không có bao bì thương

Lưu ý: Điều 1 của nghị định này là "Sửa đổi, bổ sung một số điều Nghị định số 43/2017/NĐ-CP" nên rất dài, mỗi khoản là nội dung sửa đổi cho 1 Điều trong NĐ cũ, do đó ta sẽ tách từng khoản ra thành từng chunk riêng.

In [30]:
def split_amendment_chunk(big_chunk):
    """
    Tách chunk sửa đổi lớn thành các part nhỏ và bổ sung metadata mục tiêu sửa đổi.
    Hàm này hiện tại chỉ bắt các pattern bắt đầu Khoản có dạng: "1. Sửa đổi, bổ sung ...".
    """
    # 1. Trích xuất tên Nghị định gốc bị sửa đổi từ trường ten_dieu
    ten_dieu = big_chunk.get("ten_dieu", "")
    nghi_dinh_goc = ""
    
    # Tìm cụm từ bắt đầu bằng "Nghị định số..." để lấy tên văn bản gốc
    match_goc = re.search(r"(Nghị định số\s+[\w/-]+)", ten_dieu, re.IGNORECASE)
    if match_goc:
        nghi_dinh_goc = match_goc.group(1).strip()
        
    # 2. Chuẩn bị xử lý nội dung
    noi_dung_goc = big_chunk.get("noi_dung", "")
    # Tách nội dung thành từng dòng để duyệt
    lines = noi_dung_goc.split('\n')
    
    sub_chunks = []
    current_part = None
    current_text = []
    
    # Regex bắt đầu Khoản: Ví dụ "1. Sửa đổi, bổ sung Điều 1"
    # Group 1: Số thứ tự (1, 2, 3)
    # Group 2: Nội dung hành động ("Sửa đổi, bổ sung Điều 1")
    pattern_khoan = re.compile(r"^(\d+)\.\s+(Sửa đổi, bổ sung\s+.*)")
    
    for line in lines:
        match = pattern_khoan.match(line.strip())
        
        if match:
            # Nếu đang có một part mở (do các vòng lặp trước tạo ra) -> đóng lại và lưu
            if current_part is not None:
                current_part["noi_dung"] = "\n".join(current_text).strip()
                sub_chunks.append(current_part)
            
            # Khởi tạo part mới
            so_thu_tu = match.group(1)
            hanh_dong = match.group(2) # VD: "Sửa đổi, bổ sung Điều 1"
            
            # Nhân bản toàn bộ metadata từ chunk gốc
            current_part = copy.deepcopy(big_chunk)
            
            # Cập nhật ID và thứ tự part
            current_part["chunk_id"] = f"{big_chunk['chunk_id']}_Khoản_{so_thu_tu}"
            current_part["part"] = f"part_{so_thu_tu}"
            
            # --- BỔ SUNG METADATA ---
            # Tạo trường mới (hoặc bạn có thể ghi đè vào trường 'sua_doi' đang trống)
            truong_muc_tieu = hanh_dong
            if nghi_dinh_goc:
                truong_muc_tieu = f"{hanh_dong} của {nghi_dinh_goc}"
                
            current_part["sua_doi_cho"] = truong_muc_tieu
            
            # Bắt đầu gom nội dung cho part này (bao gồm cả dòng tiêu đề)
            current_text = [line]
        else:
            # Nếu không khớp Regex và đang trong một part, tiếp tục gom dòng
            if current_part is not None:
                current_text.append(line)
            # Dòng nằm trước mục "1. Sửa đổi..." (ví dụ câu dẫn) sẽ bị bỏ qua hoặc bạn có thể gom tùy ý

    # Đóng part cuối cùng sau khi hết vòng lặp
    if current_part is not None:
        current_part["noi_dung"] = "\n".join(current_text).strip()
        sub_chunks.append(current_part)
        
    return sub_chunks

In [31]:
chunk_goc = chunks_nd111[0].copy()
sub_chunks = split_amendment_chunk(chunk_goc)
sub_chunks[0]

{'chunk_id': '111/2021/NĐ-CP_Điều_1_Khoản_1',
 'noi_dung': '1. Sửa đổi, bổ sung Điều 1\n“Điều 1. Phạm vi điều chỉnh\n1. Nghị định này quy định nội dung, cách ghi và quản lý nhà nước về nhãn đối với hàng hóa lưu thông tại Việt Nam, hàng hóa xuất khẩu, nhập khẩu.\n2. Những loại hàng hóa sau đây không thuộc phạm vi điều chỉnh của Nghị định này:\na) Bất động sản;\nb) Hàng hóa tạm nhập tái xuất; hàng hóa quá cảnh, hàng hóa chuyển khẩu; hàng hóa trung chuyển; hàng hóa nhập khẩu gửi kho ngoại quan để xuất khẩu sang nước thứ ba;\nc) Hành lý của người xuất cảnh, nhập cảnh; tài sản di chuyển;\nd) Hàng hóa bị tịch thu bán đấu giá;\nđ) Hàng hóa là thực phẩm tươi, sống, thực phẩm chế biến không có bao bì và bán trực tiếp cho người tiêu dùng;\ne) Hàng hóa là nhiên liệu, nguyên liệu (nông sản, thủy sản, khoáng sản), phế liệu (trong sản xuất, kinh doanh), vật liệu xây dựng không có bao bì và được bán trực tiếp cho người tiêu dùng;\ng) Hàng hóa là xăng dầu, khí (LPG, CNG, LNG) chất lỏng, không có bao b

In [32]:
id_can_xoa = '111/2021/NĐ-CP_Điều_1'

# Tìm vị trí (index) của chunk cũ trong list
index_can_xoa = next((i for i, chunk in enumerate(chunks_nd111) if chunk['chunk_id'] == id_can_xoa), None)

# Thay thế nếu tìm thấy
if index_can_xoa is not None:
    # Cú pháp [index : index+1] sẽ cắt đúng 1 phần tử cũ ra và thay bằng toàn bộ list sub_chunks
    chunks_nd111[index_can_xoa:index_can_xoa+1] = sub_chunks
    print(f"Đã xóa chunk {id_can_xoa} và thay bằng {len(sub_chunks)} chunk nhỏ tại vị trí {index_can_xoa}.")
else:
    print("Không tìm thấy chunk cần xóa.")

Đã xóa chunk 111/2021/NĐ-CP_Điều_1 và thay bằng 9 chunk nhỏ tại vị trí 0.


In [33]:
random.sample(chunks_nd111, 2)

[{'chunk_id': '111/2021/NĐ-CP_Điều_1_Khoản_2',
  'noi_dung': '2. Sửa đổi, bổ sung Điều 2\n“Điều 2. Đối tượng áp dụng\nNghị định này áp dụng đối với tổ chức, cá nhân sản xuất, kinh doanh hàng hóa tại Việt Nam; tổ chức, cá nhân xuất khẩu, nhập khẩu hàng hóa; cơ quan nhà nước; tổ chức, cá nhân liên quan.”;',
  'so_dieu': '1',
  'ten_dieu': 'Sửa đổi, bổ sung một số điều Nghị định số 43/2017/NĐ-CP ngày 14 tháng 4 năm 2017 của Chính phủ về nhãn hàng hóa như sau:',
  'so_chuong': '',
  'ten_chuong': '',
  'bo_luat': 'Nghị định về nhãn hàng hóa',
  'so_hieu': '111/2021/NĐ-CP',
  'ngay_ky': '09/12/2021',
  'sua_doi': '',
  'nguon': 'https://thuvienphapluat.vn/van-ban/Thuong-mai/Nghi-dinh-111-2021-ND-CP-sua-doi-Nghi-dinh-43-2017-ND-CP-497099.aspx',
  'part': 'part_2',
  'sua_doi_cho': 'Sửa đổi, bổ sung Điều 2 của Nghị định số 43/2017/NĐ-CP'},
 {'chunk_id': '111/2021/NĐ-CP_Điều_1_Khoản_6',
  'noi_dung': '6. Sửa đổi, bổ sung khoản 3 Điều 12\n“Điều 12. Tên và địa chỉ của tổ chức, cá nhân chịu trách

Lưu ý: Nghị định này cũng chưa xử lý các bảng phụ lục.

#### 1.5 Nghị định 15/2018

In [34]:
file_path_nd15 = 'data/raw_html/nd15_2018.html'
para_nd15 = read_html_paragraphs(file_path_nd15)

In [35]:
para_nd15[:20]

['CHÍNH PHỦ -------',
 'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc ---------------',
 'Số: 15/2018/NĐ-CP',
 'Hà Nội, ngày 02 tháng 02 năm 2018',
 'NGHỊ ĐỊNH',
 'QUY ĐỊNH CHI TIẾT THI HÀNH MỘT SỐ ĐIỀU CỦA LUẬT AN TOÀN THỰC PHẨM',
 'Căn cứ Luật tổ chức Chính phủ ngày 19 tháng 6 năm 2015;',
 'Căn cứ Luật an toàn thực phẩm ngày 17 tháng 6 năm 2010;',
 'Theo đề nghị của Bộ trưởng Bộ Y tế;',
 'Chính phủ ban hành Nghị định quy định chi tiết thi hành một số điều của Luật an toàn thực phẩm .',
 'Chương I',
 'QUY ĐỊNH CHUNG',
 'Điều 1. Phạm vi điều chỉnh',
 'Nghị định này quy định chi tiết thi hành một số điều của Luật an toàn thực phẩm về:',
 '1. Thủ tục tự công bố sản phẩm.',
 '2. Thủ tục đăng ký bản công bố sản phẩm.',
 '3. Bảo đảm an toàn thực phẩm biến đổi gen.',
 '4. Cấp Giấy chứng nhận cơ sở đủ điều kiện an toàn thực phẩm.',
 '5. Kiểm tra nhà nước về an toàn thực phẩm nhập khẩu, xuất khẩu.',
 '6. Ghi nhãn thực phẩm.']

In [36]:
meta_nd15 = {"bo_luat":"Nghị định quy định chi tiết thi hành một số điều luật an toàn thực phẩm",
            "so_hieu":"15/2018/NĐ-CP",
            "ngay_ky":"02/2/2018",
            "sua_doi":"",
            "nguon":"https://thuvienphapluat.vn/van-ban/The-thao-Y-te/Nghi-dinh-15-2018-ND-CP-huong-dan-Luat-an-toan-thuc-pham-341254.aspx"}

chunks_nd15 = chunk_paragraphs(para_nd15, meta_nd15)

Đã cắt bỏ phần rác cuối văn bản từ dòng: 'Nơi nhận: - Ban Bí thư Trung ương Đảng; - Thủ tướng, các Phó Thủ tướng Chính phủ; - Các bộ, cơ quan ngang bộ, cơ quan thuộc Chính phủ; - HĐND, UBND các tỉnh, thành phố trực thuộc trung ương; - Văn phòng Trung ương và các Ban của Đảng; - Văn phòng Tổng Bí thư; - Văn phòng Chủ tịch nước; - Hội đồng dân tộc và các Ủy ban của Quốc hội; - Văn phòng Quốc hội; - Tòa án nhân dân tối cao; - Viện kiểm sát nhân dân tối cao; - Kiểm toán nhà nước; - Ủy ban Giám sát tài chính Quốc gia; - Ngân hàng Chính sách xã hội; - Ngân hàng Phát triển Việt Nam; - Ủy ban trung ương Mặt trận Tổ quốc Việt Nam; - Cơ quan trung ương của các đoàn thể; - VPCP: BTCN, các PCN, Trợ lý TTg, TGĐ Cổng TTĐT, các Vụ, Cục, đơn vị trực thuộc, Công báo; - Lưu: VT, KGVX (2).'


In [37]:
length_nd15 = pd.Series([len(c['noi_dung']) for c in chunks_nd15])
length_nd15.describe()

count      44.000000
mean     1379.613636
std      1069.532961
min       226.000000
25%       474.250000
50%      1006.500000
75%      1977.500000
max      3957.000000
dtype: float64

In [38]:
random.sample(chunks_nd15, 2)

[{'chunk_id': '15/2018/NĐ-CP_Điều_31',
  'noi_dung': '1. Phụ gia thực phẩm thuộc danh mục các chất phụ gia được phép sử dụng trong thực phẩm do Bộ Y tế quy định thuộc đối tượng tự công bố.\n2. Thủ tục tự công bố sản phẩm đối với phụ gia thực phẩm đơn chất thực hiện theo quy định tại Điều 5 Nghị định này .',
  'so_dieu': '31',
  'ten_dieu': 'Quy định về phụ gia thực phẩm đơn chất',
  'so_chuong': 'X',
  'ten_chuong': 'ĐIỀU KIỆN BẢO ĐẢM AN TOÀN THỰC PHẨM TRONG SẢN XUẤT, KINH DOANH VÀ SỬ DỤNG PHỤ GIA THỰC PHẨM',
  'bo_luat': 'Nghị định quy định chi tiết thi hành một số điều luật an toàn thực phẩm',
  'so_hieu': '15/2018/NĐ-CP',
  'ngay_ky': '02/2/2018',
  'sua_doi': '',
  'nguon': 'https://thuvienphapluat.vn/van-ban/The-thao-Y-te/Nghi-dinh-15-2018-ND-CP-huong-dan-Luat-an-toan-thuc-pham-341254.aspx',
  'part': ''},
 {'chunk_id': '15/2018/NĐ-CP_Điều_5',
  'noi_dung': '1. Hồ sơ tự công bố sản phẩm bao gồm:\na) Bản tự công bố sản phẩm theo Mẫu số 01 Phụ lục I ban hành kèm theo Nghị định này;\

### 2. Xử lý các chunk dài

In [39]:
all_chunks = chunks_attp + chunks_hq + chunks_nd43 + chunks_nd111 + chunks_nd15

#### 2.1 Chọn mô hình embedding và xác định max_seq_length

In [40]:
import os
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

os.environ["HF_HUB_DISABLE_XET"] = "1"

In [41]:
model_name = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_seq_length = model.max_seq_length

print("max_seq_length của mô hình là:", max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

max_seq_length của mô hình là: 512


#### 2.2 Đếm số chunk vượt ngưỡng token

In [42]:
seq_lengths = []

for c in all_chunks:
    # Đo token thực tế bằng tokenizer của mô hình
    # Thêm truncation=False để đếm chính xác tổng số token dù có vượt giới hạn
    tokens = tokenizer(c['noi_dung'], truncation=False)['input_ids']
    
    seq_lengths.append({
        'chunk_id': c.get('chunk_id', 'Unknown'),
        'num_tokens': len(tokens)
    })

df_thong_ke = pd.DataFrame(seq_lengths)

print("=== PHÂN PHỐI SỐ LƯỢNG TOKEN ===")
print(df_thong_ke['num_tokens'].describe())
print("\n" + "="*32 + "\n")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1208 > 512). Running this sequence through the model will result in indexing errors


=== PHÂN PHỐI SỐ LƯỢNG TOKEN ===
count     257.000000
mean      263.260700
std       211.566345
min        16.000000
25%       110.000000
50%       206.000000
75%       337.000000
max      1342.000000
Name: num_tokens, dtype: float64




In [43]:
chunks_vuot_nguong = df_thong_ke[df_thong_ke['num_tokens'] > max_seq_length]
so_luong_vuot = len(chunks_vuot_nguong)

print(f"Số lượng chunk vượt quá {max_seq_length} tokens: {so_luong_vuot} chunks")

if so_luong_vuot > 0:
    print("\nDanh sách các chunk vượt ngưỡng chi tiết:")
    
    # Dùng iterrows() để lấy cả ID và số lượng token trên từng dòng
    for index, row in chunks_vuot_nguong.iterrows():
        print(f"- {row['chunk_id']}: {row['num_tokens']} tokens")

Số lượng chunk vượt quá 512 tokens: 26 chunks

Danh sách các chunk vượt ngưỡng chi tiết:
- 61/VBHN-VPQH_Điều_2: 1208 tokens
- 61/VBHN-VPQH_Điều_5: 605 tokens
- 61/VBHN-VPQH_Điều_7: 645 tokens
- 61/VBHN-VPQH_Điều_8: 559 tokens
- 61/VBHN-VPQH_Điều_55: 640 tokens
- 54/VBHN-VPQH_Điều_4: 1342 tokens
- 54/VBHN-VPQH_Điều_18: 725 tokens
- 54/VBHN-VPQH_Điều_58: 669 tokens
- 54/VBHN-VPQH_Điều_63: 805 tokens
- 54/VBHN-VPQH_Điều_74: 527 tokens
- 54/VBHN-VPQH_Điều_76: 654 tokens
- 54/VBHN-VPQH_Điều_80: 613 tokens
- 54/VBHN-VPQH_Điều_88: 647 tokens
- 43/2017/NĐ-CP_Điều_3: 877 tokens
- 43/2017/NĐ-CP_Điều_12: 524 tokens
- 43/2017/NĐ-CP_Điều_17: 569 tokens
- 111/2021/NĐ-CP_Điều_1_Khoản_5: 712 tokens
- 15/2018/NĐ-CP_Điều_3: 805 tokens
- 15/2018/NĐ-CP_Điều_5: 600 tokens
- 15/2018/NĐ-CP_Điều_7: 868 tokens
- 15/2018/NĐ-CP_Điều_8: 935 tokens
- 15/2018/NĐ-CP_Điều_19: 609 tokens
- 15/2018/NĐ-CP_Điều_22: 779 tokens
- 15/2018/NĐ-CP_Điều_27: 1015 tokens
- 15/2018/NĐ-CP_Điều_28: 755 tokens
- 15/2018/NĐ-CP_Điều_29

#### 2.3 Xử lý các chunk vượt ngưỡng

In [44]:
def split_long_chunk(chunk, max_tokens, tokenizer):
    """
    Hàm xử lý cắt chunk quá dài đệ quy, gom nhóm tham lam.
    Bổ sung: Ghép sua_doi_cho vào prefix để tăng mật độ thông tin cho Vector.
    """
    text = chunk.get('noi_dung', '')
    
    so_dieu = chunk.get('so_dieu', 'X')
    ten_dieu = chunk.get('ten_dieu', 'Không có tên')
    sua_doi_cho = chunk.get('sua_doi_cho', '').strip()
    
    # 1. Tối ưu hóa Tiền tố (Prefix) cho văn bản sửa đổi
    # Nếu chunk có trường sua_doi_cho, ưu tiên dùng nó làm tiêu đề
    if sua_doi_cho:
        prefix = f"Điều {so_dieu}. {sua_doi_cho}:\n"
    else:
        prefix = f"Điều {so_dieu}. {ten_dieu}:\n"
        
    prefix_tokens = len(tokenizer(prefix, truncation=False)['input_ids'])
    
    # 2. Xử lý chunk NGẮN (Không cần cắt)
    if len(tokenizer(prefix + text, truncation=False)['input_ids']) <= max_tokens:
        new_chunk = copy.deepcopy(chunk)
        new_chunk['parent_id'] = new_chunk['chunk_id'] 
        new_chunk['text_for_embedding'] = prefix + text
        return [new_chunk]
        
    # 3. Hàm đệ quy nội bộ để chia nhỏ và gom nhóm
    def split_and_group(content, level):
        patterns = [
            r'\n(?=\d+\.\s)',      # Regex Khoản: \n1. 
            r'\n(?=[a-zđ]\)\s)'    # Regex Điểm: \na) 
        ]
        
        if len(tokenizer(prefix + content, truncation=False)['input_ids']) <= max_tokens or level >= len(patterns):
            return [content]
            
        parts = re.split(patterns[level], content)
        
        if len(parts) <= 1:
            return split_and_group(content, level + 1)
            
        processed_parts = []
        for part in parts:
            part = part.strip()
            if not part:
                continue
            
            if len(tokenizer(prefix + part, truncation=False)['input_ids']) > max_tokens:
                processed_parts.extend(split_and_group(part, level + 1))
            else:
                processed_parts.append(part)
                
        grouped_contents = []
        current_group = []
        current_tokens = 0
        
        for p in processed_parts:
            p_toks = len(tokenizer(p, truncation=False)['input_ids'])
            
            if p_toks + prefix_tokens > max_tokens:
                if current_group:
                    grouped_contents.append("\n\n".join(current_group))
                    current_group = []
                    current_tokens = 0
                grouped_contents.append(p) 
                continue

            if current_tokens + p_toks > max_tokens - prefix_tokens and current_group:
                grouped_contents.append("\n\n".join(current_group))
                current_group = [p]
                current_tokens = p_toks
            else:
                current_group.append(p)
                current_tokens += p_toks
                
        if current_group:
            grouped_contents.append("\n\n".join(current_group))
            
        return grouped_contents

    # 4. Kích hoạt luồng xử lý từ tầng 0 (Khoản) cho chunk DÀI
    final_contents = split_and_group(text, level=0)
    
    # 5. Đóng gói lại thành danh sách sub_chunks
    sub_chunks = []
    for i, content in enumerate(final_contents):
        new_chunk = copy.deepcopy(chunk)
        new_chunk['chunk_id'] = f"{chunk.get('chunk_id')}_part_{i+1}"
        new_chunk['parent_id'] = chunk.get('chunk_id')
        new_chunk['noi_dung'] = content 
        new_chunk['text_for_embedding'] = prefix + content
        
        if 'so_khoan' in new_chunk:
            del new_chunk['so_khoan']
            
        sub_chunks.append(new_chunk)
        
    return sub_chunks

In [45]:
all_chunks_final = []
for c in all_chunks:
    all_chunks_final.extend(split_long_chunk(c, max_seq_length, tokenizer))

print(f"Tổng số chunk TRƯỚC khi cắt: {len(all_chunks)}")
print(f"Tổng số chunk SAU khi cắt: {len(all_chunks_final)}")

# Đo lại và tự kiểm tra (Sanity Check)
stats = []
for c in all_chunks_final:
    t_len = len(tokenizer(c['text_for_embedding'], truncation=False)['input_ids'])
    stats.append({'chunk_id': c['chunk_id'], 'num_tokens': t_len})

df_thong_ke = pd.DataFrame(stats)

print("\n=== PHÂN PHỐI TOKEN MỚI ===")
print(df_thong_ke['num_tokens'].describe())

Tổng số chunk TRƯỚC khi cắt: 257
Tổng số chunk SAU khi cắt: 291

=== PHÂN PHỐI TOKEN MỚI ===
count    291.000000
mean     250.037801
std      135.177082
min       23.000000
25%      130.000000
50%      230.000000
75%      357.500000
max      505.000000
Name: num_tokens, dtype: float64


In [46]:
# Tạo dictionary để tra cứu nhanh nội dung dựa trên chunk_id
dict_tra_cuu = {c['chunk_id']: c.get('text_for_embedding', '') for c in all_chunks_final}

# Lọc các chunk vượt ngưỡng
chunks_vuot_nguong = df_thong_ke[df_thong_ke['num_tokens'] > max_seq_length]
so_luong_vuot = len(chunks_vuot_nguong)

print(f"\nCẢNH BÁO: Còn {so_luong_vuot} chunk vẫn vượt > {max_seq_length} tokens.")

if so_luong_vuot > 0:
    print("Danh sách các chunk cứng đầu và chi tiết nội dung bị cắt rơi rụng:")
    
    for index, row in chunks_vuot_nguong.iterrows():
        chunk_id = row['chunk_id']
        num_tokens = row['num_tokens']
        so_token_vuot = num_tokens - max_seq_length
        
        # Gọi nội dung gốc từ dict tra cứu
        text = dict_tra_cuu.get(chunk_id, '')
        
        # Lấy danh sách token ids (không cắt)
        input_ids = tokenizer(text, truncation=False)['input_ids']
        
        # Cắt mảng để lấy phần token dư thừa và giải mã
        truncated_ids = input_ids[max_seq_length:]
        truncated_text = tokenizer.decode(truncated_ids, skip_special_tokens=True)
        
        # In kết quả
        print(f"\n[-] {chunk_id}: {num_tokens} tokens (Vượt {so_token_vuot} tokens)")
        print(f"    NỘI DUNG BỊ CẮT:\n    >>> {truncated_text} <<<")


CẢNH BÁO: Còn 0 chunk vẫn vượt > 512 tokens.


### 3. Tổng hợp và lưu dữ liệu

In [47]:
import json

print(f"Tổng số chunks sau khi gom: {len(all_chunks_final)}")

duong_dan_file = "data/chunks.json"

with open(duong_dan_file, "w", encoding="utf-8") as file:
    # Tham số ensure_ascii=False CỰC KỲ QUAN TRỌNG để tiếng Việt hiển thị đúng, 
    # không bị biến thành các mã như \u0110
    # Tham số indent=2 giúp format file JSON đẹp, có thụt lề, dễ dàng mở lên kiểm tra bằng mắt
    json.dump(all_chunks_final, file, ensure_ascii=False, indent=2)

print(f"Đã lưu thành công dữ liệu vào file: {duong_dan_file}")

Tổng số chunks sau khi gom: 291
Đã lưu thành công dữ liệu vào file: data/chunks.json
